In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
import json
import copy

In [2]:
class GraphState(TypedDict):
    messages: Annotated[list, operator.add]
    trace_log: Annotated[list, operator.add]

    counter: int
    result: str

In [3]:
# 工具函数
def pretty_print(title, data):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

    print(json.dumps(data, indent=2, ensure_ascii=False))


def simulate_merge(old_state, update):
    """
    手动模拟 merge
    用于教学观察
    """

    merged = copy.deepcopy(old_state)

    for key, value in update.items():

        # list reducer
        if isinstance(value, list) and isinstance(merged.get(key), list):
            merged[key] = merged[key] + value

        else:
            merged[key] = value

    return merged

In [4]:
def node_alpha(state: GraphState):

    pretty_print(
        "[Alpha] Received State",
        state
    )

    update = {
        "messages": ["alpha says hello"],
        "trace_log": ["alpha executed"]
    }

    pretty_print(
        "[Alpha] Returned Update",
        update
    )

    merged = simulate_merge(state, update)

    pretty_print(
        "[Alpha] Merged State",
        merged
    )

    return update


# =====================================
# Node Beta
# =====================================

def node_beta(state: GraphState):

    pretty_print(
        "[Beta] Received State",
        state
    )

    update = {
        "messages": ["beta processed message"],
        "counter": state.get("counter", 0) + 1,
        "trace_log": ["beta executed"]
    }

    pretty_print(
        "[Beta] Returned Update",
        update
    )

    merged = simulate_merge(state, update)

    pretty_print(
        "[Beta] Merged State",
        merged
    )

    return update


# =====================================
# Node Gamma
# =====================================

def node_gamma(state: GraphState):

    pretty_print(
        "[Gamma] Received State",
        state
    )

    update = {
        "result": f"Final count = {state['counter']}",
        "trace_log": ["gamma executed"]
    }

    pretty_print(
        "[Gamma] Returned Update",
        update
    )

    merged = simulate_merge(state, update)

    pretty_print(
        "[Gamma] Merged State",
        merged
    )

    return update

In [5]:
builder = StateGraph(GraphState)

builder.add_node("alpha", node_alpha)
builder.add_node("beta", node_beta)
builder.add_node("gamma", node_gamma)

builder.set_entry_point("alpha")

builder.add_edge("alpha", "beta")
builder.add_edge("beta", "gamma")
builder.add_edge("gamma", END)

graph = builder.compile()

In [6]:
initial_state = {
    "messages": [],
    "trace_log": [],
    "counter": 0,
    "result": ""
}

result = graph.invoke(initial_state)

pretty_print(
    "FINAL STATE",
    result
)


[Alpha] Received State
{
  "messages": [],
  "trace_log": [],
  "counter": 0,
  "result": ""
}

[Alpha] Returned Update
{
  "messages": [
    "alpha says hello"
  ],
  "trace_log": [
    "alpha executed"
  ]
}

[Alpha] Merged State
{
  "messages": [
    "alpha says hello"
  ],
  "trace_log": [
    "alpha executed"
  ],
  "counter": 0,
  "result": ""
}

[Beta] Received State
{
  "messages": [
    "alpha says hello"
  ],
  "trace_log": [
    "alpha executed"
  ],
  "counter": 0,
  "result": ""
}

[Beta] Returned Update
{
  "messages": [
    "beta processed message"
  ],
  "counter": 1,
  "trace_log": [
    "beta executed"
  ]
}

[Beta] Merged State
{
  "messages": [
    "alpha says hello",
    "beta processed message"
  ],
  "trace_log": [
    "alpha executed",
    "beta executed"
  ],
  "counter": 1,
  "result": ""
}

[Gamma] Received State
{
  "messages": [
    "alpha says hello",
    "beta processed message"
  ],
  "trace_log": [
    "alpha executed",
    "beta executed"
  ],
  "coun